# Train a real DBoW2 vocabulary for RootSIFT -- for the custom `SlamWorker`/`kitti_ate` pipeline

Trains `analyze/train_sift_dbow_vocabulary.cpp` (new this session, see `DEBUGGING.md` "Session 15" item 14): a real DBoW2 hierarchical-k-means vocabulary tree over **RootSIFT** descriptors, using a new `DBoW2::FClass` adapter (`third_party/DBoW2/DBoW2/FRootSift.h/.cpp`, squared-L2 distance) since the vendored DBoW2 only ever shipped `FORB.h` (ORB, Hamming distance) before now.

This targets the **custom** `SlamWorker`/`kitti_ate` pipeline (`src/vision/SlamWorker.cpp`), a completely different track from the `third_party/ORB_SLAM3_SIFT` fork the other notebooks in this folder (`cudasift_kaggle.ipynb`, `lightglue_kaggle.ipynb`) target. The trained vocabulary is meant as an **alternative** to that pipeline's existing VLAD-based loop-closure candidate search (`vision/VladVocabulary.h`), not a replacement -- VLAD stays wired exactly as-is; the new `siftdbow` CLI flag on `kitti_ate` (`argv[36]`/`argv[37]`) is purely additive.

**No GPU needed** -- this is CPU/RAM-bound (hierarchical k-means clustering over descriptor vectors), unlike `cudasift_kaggle.ipynb`. A CPU-only Kaggle session is fine; just make sure you have enough session time/RAM for the training set size you choose (see the config cell below).

**A real bug was found and fixed this session** in the vendored `TemplatedVocabulary.h`: its k-means convergence loop had no iteration cap and could spin (effectively) forever on continuous float descriptors -- fixed with a `kMaxHKmeansIterations = 50` cap (see `DEBUGGING.md` for the full writeup). Make sure the cloned repo includes that fix (it's part of the same commit as this notebook) -- without it, training will likely hang.

**Before running:**
1. Notebook settings -> Accelerator -> None (CPU is fine). Internet -> On.
2. Add Data -> attach one or more KITTI odometry (grayscale) sequence datasets -- needs `.../image_0/*.png` and `times.txt` per sequence under `/kaggle/input`. More sequences = a more general vocabulary, but also a bigger/slower training pool.
3. Set `REPO_URL` below to this project's GitHub URL (must be pushed first, including the `HKmeansStep` iteration-cap fix and the new `FRootSift`/`train_sift_dbow_vocabulary` files).

In [ ]:
# --- [1/6] Clone the repo ---
import os

REPO_URL = "https://github.com/nguyenhuunam852/ORB-SLAM-SIFT.git"  # set to your pushed repo
REPO_DIR = "/kaggle/working/ORB-SLAM-SIFT"

if not os.path.isdir(REPO_DIR):
    !git clone --depth 1 {REPO_URL} {REPO_DIR}
else:
    print(f"{REPO_DIR} already present, skipping clone (rm -rf it to force a fresh clone)")

In [ ]:
# --- [2/6] Locate every attached KITTI sequence (grayscale image_0/ + times.txt) ---
import glob, os

hits = sorted(set(os.path.dirname(p) for p in glob.glob("/kaggle/input/**/image_0", recursive=True)))
assert hits, "No image_0/ folder found under /kaggle/input -- attach at least one KITTI odometry dataset"

seq_dirs = [h for h in hits if os.path.isfile(os.path.join(h, "times.txt"))]
assert seq_dirs, "Found image_0/ folders but none had a sibling times.txt -- train_sift_dbow_vocabulary needs both"

print(f"Found {len(seq_dirs)} usable sequence dir(s):")
for d in seq_dirs:
    n = len(glob.glob(os.path.join(d, "image_0", "*.png")))
    print(f"  {d}  ({n} images)")

In [ ]:
# --- [3/6] Install build dependencies ---
# Deliberately NOT using the project's top-level CMakeLists.txt here -- that
# unconditionally find_package()s Qt6/Ceres/g2o/the full vendored ORB_SLAM3
# for the GUI app and every analyze/ tool, none of which
# train_sift_dbow_vocabulary actually needs (it only links FeatureDetector.cpp
# + the DBoW2 sources against OpenCV). Compiling it directly with g++ instead
# keeps this notebook fast and avoids installing a pile of unrelated, heavy
# dependencies on a CPU-only Kaggle session. libboost-dev is DBoW2's own
# BowVector.h/FeatureVector.h needing <boost/serialization/...> HEADERS only
# (no library link, see CMakeLists.txt's own comment on this) for an optional
# (de)serialization path this build never calls.
!apt-get -qq update && apt-get -qq install -y libopencv-dev libboost-dev pkg-config g++ > /dev/null
!pkg-config --modversion opencv4

In [ ]:
# --- [4/6] Compile train_sift_dbow_vocabulary ---
import subprocess

os.chdir(REPO_DIR)

sources = [
    "analyze/train_sift_dbow_vocabulary.cpp",
    "src/vision/FeatureDetector.cpp",
    "third_party/DBoW2/DBoW2/BowVector.cpp",
    "third_party/DBoW2/DBoW2/FRootSift.cpp",
    "third_party/DBoW2/DBoW2/FeatureVector.cpp",
    "third_party/DBoW2/DBoW2/ScoringObject.cpp",
    "third_party/DBoW2/DUtils/Random.cpp",
    "third_party/DBoW2/DUtils/Timestamp.cpp",
]

opencv_cflags = subprocess.check_output(["pkg-config", "--cflags", "opencv4"]).decode().split()
opencv_libs = subprocess.check_output(["pkg-config", "--libs", "opencv4"]).decode().split()

cmd = (
    ["g++", "-O3", "-std=c++17", "-Isrc", "-Ithird_party/DBoW2"]
    + opencv_cflags
    + sources
    + opencv_libs
    + ["-o", "train_sift_dbow_vocabulary"]
)
print(" ".join(cmd))
subprocess.run(cmd, check=True)
print("build OK")

In [ ]:
# --- [5/6] Configure and run training ---
# k/L: branching factor / tree depth -- k^L words (must be k<=20, L<=10, see
# TemplatedVocabulary::loadFromTextFile()'s own sanity bounds). k=10,L=5
# below (100k words) is a real-sized vocabulary, smaller than ORB-SLAM3's own
# ORBvoc.txt (k=10,L=6, 1M words) -- start here and try L=6 in a later run
# once you've seen how long L=5 actually takes on your attached data (this
# hasn't been measured at this scale yet, only smoke-tested at a tiny
# k=8,L=3 locally -- 30s for 91 images/170k descriptors AFTER the
# iteration-cap fix, unusably slow before it -- so treat the real-size
# timing here as new information, not an established estimate).
#
# FRAME_STRIDE: only extract from every Nth frame. This is the main lever
# for keeping the training pool (and therefore wall-clock time) bounded --
# DBoW2's hierarchical k-means is far more expensive than VLAD's flat
# single-pass k-means (see analyze/orbslam3_vlad_train.cpp for comparison),
# so err on the side of a larger stride for a first run.
K = 10
L = 5
FRAME_STRIDE = 5
OUT_PATH = "/kaggle/working/sift_dbow_vocab.txt"

import time
t0 = time.time()
cmd = ["./train_sift_dbow_vocabulary", OUT_PATH, str(K), str(L), str(FRAME_STRIDE)] + seq_dirs
print(" ".join(cmd))
subprocess.run(cmd, check=True)
print(f"total wall time: {time.time() - t0:.1f}s")

In [ ]:
# --- [6/6] Confirm the output vocabulary is ready to download ---
import os

size_mb = os.path.getsize(OUT_PATH) / 1e6
print(f"{OUT_PATH}: {size_mb:.1f} MB")
print()
print("Download this file (Kaggle's Output panel, or /kaggle/working/sift_dbow_vocab.txt),")
print("place it anywhere in the project (e.g. vocabulary_sift/), then run kitti_ate with:")
print("  ... siftdbow <path-to-this-file>")
print("(argv[36]/argv[37] -- see kitti_ate's own usage text). Compare against the existing")
print("'vlad vocabulary_sift/vlad_codebook_all_rootsift.yml' flag on the SAME baseline config")
print("(currently 72.550m ATE, see DEBUGGING.md Session 15 item 10) for a fair single-variable comparison.")